# FitStreak: Data Science Pipeline Walkthrough
### Course: Data Science Applications (DSA) Mini-Project
**Team**: Ashwith Ashok Shetty, Shivam Swapnil Revankar
**Dataset**: Kaggle Gym Members Exercise Dataset (`valakhorasani/gym-members-exercise-dataset`)

This notebook walks through the 5 core stages of the FitStreak pipeline:
1. **Data Loading & Cleaning**: Ingest with Polars, check schema, summary stats, and missing values.
2. **Outlier Detection (IQR Method)**: Interquartile range analysis on numeric indicators.
3. **Correlation & PCA**: Pearson correlation matrix, multicollinearity checks, eigenvalues and scree plot.
4. **Supervised ML Models**: Simple Linear Regression, Multiple Linear Regression, and Logistic Regression.
5. **Serialization**: Model evaluation and artifact export with `joblib` for FastAPI serving.

## Stage 1: Data Ingestion with Polars

In [1]:
import polars as pl
import numpy as np
import joblib
from pathlib import Path

DATA_PATH = Path('../data/gym_members_exercise_tracking.csv')
df = pl.read_csv(DATA_PATH)
print(f'Dataset Shape: {df.shape}')
print('Columns and Dtypes:')
for col, dtype in df.schema.items():
    print(f'  - {col}: {dtype}')

## Stage 2: Missing Values & Outlier Detection (IQR Method)

In [2]:
numeric_cols = [
    'Age', 'Weight (kg)', 'Height (m)', 'Max_BPM', 'Avg_BPM', 'Resting_BPM',
    'Session_Duration (hours)', 'Calories_Burned', 'Fat_Percentage',
    'Water_Intake (liters)', 'Workout_Frequency (days/week)', 'BMI'
]
print('Missing Values Check:')
null_counts = {col: df[col].null_count() for col in df.columns}
print(null_counts)

print('\nOutlier Detection (IQR Method):')
outlier_summary = {}
for col in numeric_cols:
    q1 = float(df[col].quantile(0.25))
    q3 = float(df[col].quantile(0.75))
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_count = df.filter((pl.col(col) < lower) | (pl.col(col) > upper)).height
    outlier_summary[col] = {'q1': q1, 'q3': q3, 'iqr': iqr, 'outliers': outlier_count}
    print(f'  {col}: IQR={iqr:.2f}, Bounds=[{lower:.2f}, {upper:.2f}], Outliers={outlier_count}')

## Stage 3: Correlation & PCA Dimensionality Reduction

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X_num = df.select(numeric_cols).to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_num)

pca = PCA()
pca.fit(X_scaled)

print('Explained Variance Ratios (Scree Plot components):')
for i, ratio in enumerate(pca.explained_variance_ratio_):
    cum = sum(pca.explained_variance_ratio_[:i+1])
    print(f'  PC{i+1}: {ratio*100:.2f}% (Cumulative: {cum*100:.2f}%)')

## Stage 4: Supervised ML Models

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import r2_score, mean_squared_error, accuracy_score, confusion_matrix

cat_features = ['Gender', 'Workout_Type']
cat_matrix = np.column_stack([df[c].to_list() for c in cat_features])
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_cats = encoder.fit_transform(cat_matrix)
X_all = np.hstack([X_num, encoded_cats])

y_cal = df['Calories_Burned'].to_numpy()
y_lvl = df['Experience_Level'].to_numpy()

# Model 1: Simple Linear Regression
X_dur = df.select(['Session_Duration (hours)']).to_numpy()
slr = LinearRegression().fit(X_dur, y_cal)
slr_r2 = r2_score(y_cal, slr.predict(X_dur))
print(f'Simple LR: Calories = {slr.coef_[0]:.2f} * Duration + {slr.intercept_:.2f} | R² = {slr_r2:.4f}')

# Model 2: Multiple Linear Regression
mlr = LinearRegression().fit(X_all, y_cal)
mlr_r2 = r2_score(y_cal, mlr.predict(X_all))
rmse = np.sqrt(mean_squared_error(y_cal, mlr.predict(X_all)))
print(f'Multiple LR: R² = {mlr_r2:.4f}, RMSE = {rmse:.2f}')

# Model 3: Logistic Regression
clf = LogisticRegression(max_iter=1000).fit(StandardScaler().fit_transform(X_all), y_lvl)
acc = accuracy_score(y_lvl, clf.predict(StandardScaler().fit_transform(X_all)))
print(f'Logistic Regression Accuracy: {acc*100:.2f}%')

## Stage 5: Artifact Verification

In [5]:
models_dir = Path('../models')
for p in sorted(models_dir.glob('*')):
    print(f'  Found serialized artifact: {p.name} ({p.stat().st_size:,} bytes)')
print('Ready for FastAPI integration!')